In [ ]:
import sys
import os
import tensorflow as tf
import pandas as pd
import nrrd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.colors import ListedColormap
sys.path.insert(0, os.path.abspath('..'))
import resample
from collections import namedtuple
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import classification_report
from sklearn.metrics import jaccard_score
from scipy import ndimage

In [ ]:
df = pd.read_csv('/work/jprieto/data/remote/EGower/jprieto/eyes_cropped_resampled_512_seg_test_sev.csv')
print(df)

In [ ]:
model = tf.keras.models.load_model('/work/jprieto/data/remote/EGower/jprieto/trained/eyes_cropped_resampled_512_seg_train_08282021', custom_objects={'tf': tf})
model.summary()

In [ ]:
row = df.iloc[33]
y_pred_arr = []
y_true_arr = []
dice_arr = []

for idx, row in df.iterrows():
    img = row["img"]
    seg = row["seg"]
    print(img)
    img_np, header = nrrd.read(img, index_order='C')
    y_true, header = nrrd.read(seg, index_order='C')
    
    y_pred = model.predict(np.expand_dims(img_np, axis=0))
    
    y_pred = np.reshape(y_pred, -1)
    y_true = np.reshape(y_true, -1)
    
    jaccard = jaccard_score(y_true, y_pred, average=None)
    dice = 2.0*jaccard/(1.0 + jaccard)
    if(len(dice) == 4):
        print(dice)
        dice_arr.append(dice)
    else:
        dice_arr.append([0, 0, 0, 0])
        print("WTF!", row["img"])

In [ ]:
df["dice_arr"] = dice_arr
df_group_sev = df.groupby("severity_class")

fig, axs = plt.subplots(2,2, figsize=(14,14))
for k in df_group_sev.groups.keys():
    df_sev = df_group_sev.get_group(k)
    d_arr = list(df_sev["dice_arr"])
    d_arr = np.array(d_arr)
    ax = axs[int(k/2),int(k%2)]
    ax.set_title('sev' + str(k))
    s = sns.violinplot(data=d_arr, cut=0, scale="count", ax=ax)

In [ ]:

y_pred = model.predict(np.expand_dims(img_np, axis=0))

label_num = 3
size = 32
neigborhood = int(size/2)
num_samples = 32

seg_np = np.squeeze(prediction[0], axis=-1)
y, x = np.where(seg_np == label_num)
z = np.polyfit(x, y, 3)
poly = np.poly1d(z)

min_x = np.min(x) + neigborhood
max_x = np.max(x) - neigborhood
max_y = np.max(y) - neigborhood
dx = (max_x - min_x)/num_samples

out_stack = []
x_values = []
y_values = []
for i in range(num_samples):
    x = min_x + i*dx
    y = poly(x)

    x_values.append(x)
    y_values.append(y)

    start_x = min(max(int(x) - neigborhood, 0), max_x  - neigborhood)
    start_y = min(max(int(y) - neigborhood, 0), max_y  - neigborhood)

    end_x = start_x + size
    end_y = start_y + size


    out_stack.append(img_np[start_y:end_y,start_x:end_x,:])

In [ ]:
row = df.iloc[0]
img = row["img"]
seg = row["seg"]
print(img)
img_np, header = nrrd.read(img, index_order='C')
seg_np, header = nrrd.read(seg, index_order='C')

degree = np.random.random() * 180 - 90 
img_np = ndimage.rotate(img_np, degree, reshape=False, 
                mode='reflect')
seg_np = ndimage.rotate(seg_np, degree, reshape=False,
                mode='constant', order=0, cval=0)

cmap = ListedColormap(["black", "tab:red", "tab:green", "tab:blue"])
plt.figure(figsize = (15,15))
plt.imshow(img_np)
plt.imshow(seg_np, cmap=cmap, interpolation='nearest', alpha=0.4)
# plt.plot(x_values, y_values, linewidth=2, color="red")
plt.show()

In [ ]:
df1 = pd.read_csv("/work/jprieto/data/remote/EGower/jprieto/eyes_cropped_resampled_512_seg_test.csv")
df2 = pd.read_csv("/work/jprieto/data/remote/EGower/jprieto/trachoma_normals_healthy_sev123_05182021_prediction_09072021.csv")
df = df1.merge(df2, on='img')
print(df1)
print(df2)
print(df)


In [ ]:
y_pred_arr = []
y_true_arr = []
dice_arr = []

for idx, row in df.iterrows():
    seg = row["seg"]
    pred = row["prediction"]
    pred_np, header = nrrd.read(pred, index_order='C')
    seg_np, header = nrrd.read(seg, index_order='C')
    
    pred_np = np.reshape(pred_np, -1)
    seg_np = np.reshape(seg_np, -1)
    
    jaccard = jaccard_score(seg_np, pred_np, average=None)
    dice = 2.0*jaccard/(1.0 + jaccard)
    if(len(dice) == 4):
        dice_arr.append(dice)
    else:
        dice_arr.append([0, 0, 0, 0])
        print("WTF!", row["img"])

In [ ]:
df["dice_arr"] = dice_arr
df_group_sev = df.groupby("severity_class")

fig, axs = plt.subplots(2,2, figsize=(14,14))
for k in df_group_sev.groups.keys():
    df_sev = df_group_sev.get_group(k)
    d_arr = list(df_sev["dice_arr"])
    d_arr = np.array(d_arr)
    ax = axs[int(k/2),int(k%2)]
    ax.set_title('sev' + str(k))
    ax.set_ylim([0, 1])
    s = sns.violinplot(data=d_arr, cut=0, scale="count", ax=ax)

In [ ]:
for k in df_group_sev.groups.keys():
    df_sev = df_group_sev.get_group(k)
    d_arr = list(df_sev["dice_arr"])
    d_arr = np.array(d_arr)
    print("k:", k, "mean:", np.mean(d_arr, axis=0), "std:", np.std(d_arr, axis=0))
    
d_arr = list(df["dice_arr"].apply(list))
print("mean:", np.mean(d_arr, axis=0), "std:", np.std(d_arr, axis=0))